In [1]:
%autosave 60

Autosaving every 60 seconds


# RAG: Retrieval-Augmented Generation

**RAG** — подход, при котором к запросу пользователя подмешивается релевантный контекст из вашей базы документов. Модель не «знает» содержимое документов из обучения, но получает нужные фрагменты при ответе.

Зачем нужен RAG:
- **Ограничения LLM**: модель не имеет доступа к вашим данным и актуальной информации после даты обучения.
- **Актуальность**: документы можно обновлять без переобучения модели.
- **Проверяемость**: ответ опирается на извлечённые фрагменты, можно свериться с источником.


![RAG схема](img/rag_img.jpg)

## Настройка окружения

Загружаем переменные из `.env`, подключаем модели из `config`, инициализируем LLM (OpenRouter) и эмбеддинги.

In [2]:
import os
import sys
from pathlib import Path

# Корень проекта (работает при запуске из корня репо или из notebooks_rag)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Загрузка .env из корня проекта
env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    from dotenv import load_dotenv
    load_dotenv(env_path)

from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from config import model_llm, model_embedding

# Пути к данным
PATH_PDF = PROJECT_ROOT / "data" / "text" / "data_v2.pdf"
PATH_TXT = PROJECT_ROOT / "data" / "audio" / "новое_производство_газотурбинного_оборудования.txt"

# LLM через OpenRouter
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model=model_llm,
    default_headers={"X-Title": "RAG Initial Notebook"},
)

# Эмбеддинги для векторного поиска
embeddings = HuggingFaceEmbeddings(model_name=model_embedding)

print("Окружение готово. Модель LLM:", model_llm, "| Эмбеддинги:", model_embedding)

/Users/maxim/Desktop/STUDY/ai_agents_course/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1907.03it/s, Materializing param=pooler.dense.weight]                               


Окружение готово. Модель LLM: google/gemini-3.1-flash-lite-preview | Эмбеддинги: baai/bge-m3


## 1. Общее описание данных

Используем два источника:
- **PDF** (`data/text/data_v2.pdf`) — текстовый документ.
- **TXT** (`data/audio/новое_производство_газотурбинного_оборудования.txt`) — расшифровка/статья о промышленных новостях (заводы, производство).

In [3]:
# PDF
loader_pdf = PyPDFLoader(str(PATH_PDF))
docs_pdf = loader_pdf.load()
n_pages = len(docs_pdf)
text_pdf = "".join(d.page_content for d in docs_pdf)
len_pdf = len(text_pdf)

print("=== PDF (data_v2.pdf) ===")
print(f"Страниц: {n_pages}, символов: {len_pdf}")
print("\nПревью (первые 600 символов):")
print("-" * 50)
print(text_pdf[:600].strip() or "(пусто)")
print("-" * 50)

=== PDF (data_v2.pdf) ===
Страниц: 287, символов: 962365

Превью (первые 600 символов):
--------------------------------------------------
Приказ Ростехнадзора от 08.12.2020 N 505
"Об утверждении Федеральных норм и правил
в области промышленной безопасности
"Правила безопасности при ведении горных
работ и переработке твердых полезных
ископаемых"
(Зарегистрировано в Минюсте России
21.12.2020 N 61651)
Документ предоставленКонсультантПлюс
www.consultant.ru
Дата сохранения: 21.02.2024Зарегистрировано в Минюсте России 21 декабря 2020 г. N 61651
ФЕДЕРАЛЬНАЯ СЛУЖБА ПО ЭКОЛОГИЧЕСКОМУ, ТЕХНОЛОГИЧЕСКОМУ
И АТОМНОМУ НАДЗОРУ
ПРИКАЗ
от 8 декабря 2020 г. N 505
ОБ УТВЕРЖДЕНИИ ФЕДЕРАЛЬНЫХ НОРМ И ПРАВИЛ
В ОБЛАСТИ ПРОМЫШЛЕННОЙ БЕЗОПАСНОСТИ "ПРАВИЛА
--------------------------------------------------


In [4]:
# TXT
loader_txt = TextLoader(str(PATH_TXT), encoding="utf-8")
docs_txt = loader_txt.load()
text_txt = docs_txt[0].page_content if docs_txt else ""
lines_txt = text_txt.count("\n") + (1 if text_txt else 0)

print("=== TXT (новое_производство_газотурбинного_оборудования.txt) ===")
print(f"Строк/абзацев: ~{lines_txt}, символов: {len(text_txt)}")
print("\nПревью (первые 800 символов):")
print("-" * 50)
print(text_txt[:800].strip())
print("-" * 50)

=== TXT (новое_производство_газотурбинного_оборудования.txt) ===
Строк/абзацев: ~67, символов: 5811

Превью (первые 800 символов):
--------------------------------------------------
Новое высокотехнологичное предприятие по изготовлению и производству газотурбинного
оборудования открыли АМКОР в промышленном парке Турбина города Зеленодольск,
республики Татарстан. Об этом сообщили в пресс-службе главы региона.
Новый завод оснащен уникальным оборудованием. На производстве есть комплексы лазерной наплавки
и 3D-сканирования. Здесь же действует собственная лаборатория неразрушающего контроля.
Мощности предприятия предоставляют возможность выполнять полный цикл работ. Они предполагают
восстановление и производство критических компонентов горячего тракта для турбин
General Electric и Siemens. Новое предприятие стало первым и очень важным шагом к локализации
производства по этапам восстановительного ремонта, обслуживания и комплектации газотурбинных
установок. Компания IncomPr
-----------------

## 2. Вопрос к LLM без RAG

Задаём вопрос по факту, который **есть в наших данных**. Модель не имеет доступа к документам — только к своим весам. Ожидаемо: ответ будет неточным или «не знаю». Так мы показываем: **LLM без контекста не знает, RAG дальше даст контекст и правильный ответ.**

In [5]:
# Вопрос по факту из TXT (в тексте: завод АМКОР открыли в Зеленодольске)
question_txt = "В каком городе компания АМКОР открыла завод по производству газотурбинного оборудования?"
answer_txt = llm.invoke(question_txt).content

print("Источник: TXT (промышленные новости)")
print("Вопрос:", question_txt)
print("Ответ LLM без RAG:", answer_txt)

Источник: TXT (промышленные новости)
Вопрос: В каком городе компания АМКОР открыла завод по производству газотурбинного оборудования?
Ответ LLM без RAG: Компания «АМКОР» (ООО «АМКОР») открыла завод по производству газотурбинного оборудования в **городе Рыбинске** Ярославской области (на базе площадей НПО «Сатурн»).

Основная специализация предприятия — производство компонентов для газотурбинных двигателей, в том числе для нужд энергетики и авиации.


In [7]:
# Вопрос по факту из PDF (ответ есть только в документе)
# "Для отгрузки руды в забое необходимо использовать погрузочно-доставочные машины с дистанционным управлением или с расположением кабины машиниста, снабженной защитным ограждением, не ближе 4 м от переднего края ковша или другого погрузочного органа.",

question_pdf = "Какие требования ставятся к оборудованию для отгрузки руды в забое?"
answer_pdf = llm.invoke(question_pdf).content

print("Источник: PDF (data_v2.pdf)")
print("Вопрос:", question_pdf)
print("Ответ LLM без RAG:", answer_pdf)

Источник: PDF (data_v2.pdf)
Вопрос: Какие требования ставятся к оборудованию для отгрузки руды в забое?
Ответ LLM без RAG: Требования к оборудованию для отгрузки (погрузки) руды в забое определяются спецификой горно-геологических условий, объемом добычи, физико-механическими свойствами горной породы и системой разработки месторождения.

Можно выделить основные группы требований:

### 1. Производительность и мощность
*   **Соответствие темпам добычи:** Производительность погрузочного оборудования должна соответствовать пропускной способности транспортной цепочки (самосвалов, конвейеров или вагонеток) и темпам подвигания забоя.
*   **Вместимость ковша:** Размеры ковша должны быть соразмерны с габаритами транспортных средств (обычно 3–5 проходов для наполнения самосвала), чтобы минимизировать время простоя транспорта.

### 2. Горно-технические требования (геометрия)
*   **Габариты оборудования:** Оборудование должно свободно размещаться в пределах выработки, обеспечивая при этом безопасны

### Мотивация использовать RAG в работе с LLM 
![RAG схема](img/bar.png)


На рисунке показано, что чем более **популярна** сущность, о которой задают вопрос (часто встречается и просматривается, например, в Википедии), тем лучше LLM модели (в данном случае GPT‑3) отвечает на такие вопросы “из головы”, без внешних документов; для непопулярных, “длиннохвостых” сущностей качество резко падает, и тогда подключение retrieved passages (найденных текстов) существенно улучшает ответы, поэтому авторы вводят Adaptive Retrieval: система по простому порогу популярности (красная линия) решает, когда вообще не делать запрос к внешней базе (оранжевые столбцы, популярные случаи, где модель уже всё запомнила) и когда, наоборот, обязательно подмешивать внешние тексты, чтобы компенсировать слабую параметрическую память на редких фактах и одновременно не тратить ресурсы и не портить ответы там, где модель и так уверенно справляется сама. [arxiv](https://arxiv.org/pdf/2212.10511.pdf)

## 3. LangChain RAG для первого источника (PDF)

Загружаем PDF, разбиваем на чанки, строим векторное хранилище (Chroma), ретривер и цепочку: запрос → поиск релевантных фрагментов → контекст в промпт → ответ LLM.

In [8]:
# Загрузка и разбиение PDF
docs_pdf_rag = loader_pdf.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits_pdf = text_splitter.split_documents(docs_pdf_rag)

# Векторная база Chroma для PDF
vectorstore_pdf = Chroma.from_documents(
    documents=splits_pdf,
    embedding=embeddings,
    collection_name="rag_initial_pdf",
)
retriever_pdf = vectorstore_pdf.as_retriever(search_kwargs={"k": 3})

# Промпт и цепочка RAG
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

system_prompt = (
    "Ты — полезный ИИ-ассистент. Используй следующий контекст для ответа на вопрос пользователя. "
    "Если ответа в контексте нет, так и скажи.\n\nКонтекст: {context}"
)
prompt_rag = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

rag_chain_pdf = (
    {"context": retriever_pdf | format_docs, "question": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)
print("Чанков из PDF:", len(splits_pdf))

Чанков из PDF: 1300


In [9]:
# Вопрос по содержимому PDF
# "Для отгрузки руды в забое необходимо использовать погрузочно-доставочные машины с дистанционным управлением или с расположением кабины машиниста, снабженной защитным ограждением, не ближе 4 м от переднего края ковша или другого погрузочного органа."

q_pdf = "Какие требования ставятся к оборудованию для отгрузки руды в забое?"
retrieved_pdf = retriever_pdf.invoke(q_pdf)
print("Найденные фрагменты (топ-3):")
for i, doc in enumerate(retrieved_pdf, 1):
    print(f"\n--- Фрагмент {i} ---\n{doc.page_content[:400]}...")
print("\n" + "=" * 50)
answer_rag_pdf = rag_chain_pdf.invoke(q_pdf)
print("Ответ RAG (PDF):", answer_rag_pdf)

Найденные фрагменты (топ-3):

--- Фрагмент 1 ---
заколообразования.
939. Отгрузку руды (породы) в забое необходимо производить погрузочно-доставочными машинами
с дистанционным управлением или с расположением кабины машиниста, снабженной защитным
ограждением, не ближе 4 м от переднего края ковша или другого погрузочного органа.
КонсультантПлюс
надежная правовая поддержка www.consultant.ru Страница  171 из 286
Документ предоставленКонсультантПлюс
...

--- Фрагмент 2 ---
выполняться с боковых сторон.
1288. При подаче руды автотранспортом на разгрузочной площадке приемного бункера должны
быть предусмотрены упоры, исключающие скатывание автомашин в бункер, а также ограждения или
должен отсыпаться породный бруствер высотой не менее 1 м по периметру разворотных площадок у
приемных бункеров. Запрещается нахождение людей и производство каких-либо работ на разгрузочной
п...

--- Фрагмент 3 ---
при подходе очистного забоя к передовой выработке или к выработанному пространству
необходимо производи

## 4. LangChain RAG для второго источника (TXT)

Аналогичный пайплайн для текстового файла с промышленными новостями: загрузка, чанки, Chroma, ретривер, та же схема промпта и цепочки.

In [10]:
# Загрузка и разбиение TXT
docs_txt_rag = loader_txt.load()
splits_txt = text_splitter.split_documents(docs_txt_rag)

# Векторная база Chroma для TXT
vectorstore_txt = Chroma.from_documents(
    documents=splits_txt,
    embedding=embeddings,
    collection_name="rag_initial_txt",
)
retriever_txt = vectorstore_txt.as_retriever(search_kwargs={"k": 3})

rag_chain_txt = (
    {"context": retriever_txt | format_docs, "question": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)
print("Чанков из TXT:", len(splits_txt))

Чанков из TXT: 8


In [11]:
# Вопрос по факту из TXT (в тексте: Зеленодольск)
q_txt = "В каком городе компания АМКОР открыла завод по производству газотурбинного оборудования?"
retrieved_txt = retriever_txt.invoke(q_txt)
print("Найденные фрагменты (топ-3):")
for i, doc in enumerate(retrieved_txt, 1):
    print(f"\n--- Фрагмент {i} ---\n{doc.page_content[:400]}...")
print("\n" + "=" * 50)
answer_rag_txt = rag_chain_txt.invoke(q_txt)
print("Ответ RAG (TXT):", answer_rag_txt)

Найденные фрагменты (топ-3):

--- Фрагмент 1 ---
Новое высокотехнологичное предприятие по изготовлению и производству газотурбинного
оборудования открыли АМКОР в промышленном парке Турбина города Зеленодольск,
республики Татарстан. Об этом сообщили в пресс-службе главы региона.
Новый завод оснащен уникальным оборудованием. На производстве есть комплексы лазерной наплавки
и 3D-сканирования. Здесь же действует собственная лаборатория неразрушающег...

--- Фрагмент 2 ---
полимерных и строительных материалов. ТЭА-1 применяют также при производстве стекловолокна и продукции
на его основе, которые в свою очередь используют при изготовлении спецтехники, судов, самолетов,
вертолетов и другой летательной техники. Локализация производства продукции
составляет 100%. Используется российское сырье.
Третье в структуре холдинга «Атомстройкомплекс» специализированное предприят...

--- Фрагмент 3 ---
российского производства без примесей вторичного металла, то есть из зеленого алюминия с низким
углерод

## Итог

- **Без RAG**: на вопросы по фактам из PDF и TXT модель отвечала без доступа к документам — ответ неточный или «не знаю».
- **С RAG**: те же вопросы задаются после поиска по векторной базе; в промпт попадают релевантные чанки, и ответ опирается на содержимое документов.